# 💡 Notebook 7: Business Recommendations

This final notebook translates model results into actionable business decisions.

We cover:
1. Model selection rationale
2. Optimal decision threshold
3. Deployment architecture
4. Expected financial impact
5. Monitoring & retraining strategy
6. Limitations & risks


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#444', 'grid.color': '#2a2a2a',
})

print("Business Recommendations Notebook — PaySim Fraud Detection")
print("=" * 60)


## 1. 🏆 Model Selection: LightGBM

**Recommended Model: LightGBM**

| Criterion | LightGBM | XGBoost | Random Forest | Logistic Regression |
|---|---|---|---|---|
| ROC-AUC | ✅ Best | ✅ Close | Good | Baseline |
| Inference Speed | ✅ Fastest | Fast | Medium | Fastest |
| Memory Usage | ✅ Lowest | Medium | High | Lowest |
| Explainability | Good (SHAP) | Good (SHAP) | Medium | ✅ Best |
| Production Ready | ✅ Yes | Yes | Yes | Yes |

**Decision:** Deploy LightGBM as the primary scoring model.  
Keep Logistic Regression available for audit/explanation when required by regulators.


In [ ]:
# ── Threshold Sensitivity Analysis ───────────────────────────────────────
import sys
sys.path.insert(0, '..')
from src.evaluate import compute_business_roi

eval_data = pickle.load(open('../models/eval_data.pkl', 'rb'))
y_test = eval_data['y_test']
lgbm_prob = eval_data['probs']['LightGBM']

thresholds = np.linspace(0.05, 0.95, 100)
metrics = []
for t in thresholds:
    y_pred = (lgbm_prob >= t).astype(int)
    from sklearn.metrics import f1_score, recall_score, precision_score
    metrics.append({
        'threshold': t,
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
    })

thresh_df = pd.DataFrame(metrics)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresh_df['threshold'], thresh_df['f1'], color='#76B7B2', lw=2.5, label='F1 Score')
ax.plot(thresh_df['threshold'], thresh_df['recall'], color='#E15759', lw=2.5, label='Recall (Fraud Detection Rate)')
ax.plot(thresh_df['threshold'], thresh_df['precision'], color='#4E79A7', lw=2.5, label='Precision')

# Recommended threshold
rec_thresh = 0.3
ax.axvline(x=rec_thresh, color='#F28E2B', lw=2, linestyle='--', label=f'Recommended Threshold ({rec_thresh})')

ax.set_xlabel('Decision Threshold', color='white', fontsize=12)
ax.set_ylabel('Score', color='white', fontsize=12)
ax.set_title('LightGBM — Threshold Sensitivity Analysis', color='white', fontsize=14)
ax.legend(facecolor='#1a1d27', labelcolor='white', fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../visuals/07_threshold_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

optimal_row = thresh_df.loc[thresh_df['f1'].idxmax()]
print(f"Optimal F1 threshold: {optimal_row['threshold']:.2f}")
print(f"  F1: {optimal_row['f1']:.3f} | Recall: {optimal_row['recall']:.3f} | Precision: {optimal_row['precision']:.3f}")


## 2. 🎛️ Decision Threshold: 0.30 (not 0.50)

The default 0.50 threshold is wrong for fraud detection because:
- **Fraud cost >> false alarm cost** (ratio ~17,000:1)
- A 0.30 threshold catches ~15% more fraud at the cost of ~2x more false alarms
- At $856K average fraud value vs. $50 investigation cost, this is a clear win

**Rule:** Flag transaction if `P(fraud) >= 0.30`

## 3. 🏗️ Deployment Architecture

```
Transaction Request
       │
       ▼
  Feature Engineering Service (Python, <5ms)
       │
       ▼
  LightGBM Scoring API (FastAPI, <10ms)
       │
       ├─── P < 0.30 ──→ ✅ APPROVE
       │
       ├─── 0.30 ≤ P < 0.70 ──→ ⚠️ FLAG (auto-hold, SMS verification)
       │
       └─── P ≥ 0.70 ──→ 🚫 BLOCK (analyst review for high-value)
```

**Total latency target:** < 100ms end-to-end


In [ ]:
# ── Expected Annual Financial Impact ─────────────────────────────────────
# Based on PaySim proportions scaled to hypothetical $1B annual transaction volume

total_annual_transactions = 6_362_620  # PaySim simulation
fraud_transactions = 8_213             # actual fraud in dataset
avg_fraud_amount = 856_000             # average fraud USD

# Current (no ML) state
baseline_loss = fraud_transactions * avg_fraud_amount

# With LightGBM @ 0.30 threshold
recall_lgbm = 0.89
fp_rate = 0.004  # 0.4% false positive rate
total_legit = total_annual_transactions - fraud_transactions

fraud_caught = int(fraud_transactions * recall_lgbm)
fraud_missed = fraud_transactions - fraud_caught
false_alarms = int(total_legit * fp_rate)

savings = fraud_caught * avg_fraud_amount
investigation_cost = (fraud_caught + false_alarms) * 50
net_benefit = savings - investigation_cost

print("="*60)
print("  PROJECTED ANNUAL FINANCIAL IMPACT (PaySim scale)")
print("="*60)
print(f"  Total transactions:     {total_annual_transactions:>12,}")
print(f"  Fraud transactions:     {fraud_transactions:>12,}")
print(f"  Fraud caught (89%):     {fraud_caught:>12,}")
print(f"  Fraud missed:           {fraud_missed:>12,}")
print(f"  False alarms:           {false_alarms:>12,}")
print(f"")
print(f"  Fraud prevented:        ${savings:>14,.0f}")
print(f"  Investigation costs:    ${investigation_cost:>14,.0f}")
print(f"  Net benefit:            ${net_benefit:>14,.0f}")
print(f"  ROI:                    {net_benefit/investigation_cost*100:>12.0f}%")
print("="*60)


## 4. 📅 Monitoring & Retraining Strategy

| Metric | Monitoring Frequency | Alert Threshold |
|---|---|---|
| Model ROC-AUC on recent data | Weekly | Drop > 0.02 |
| Fraud rate in flagged transactions | Daily | < 2% (too many false alarms) |
| Fraud detection rate | Daily | < 80% |
| Feature distribution drift | Weekly | KS test p < 0.05 |
| Model latency | Real-time | > 150ms p99 |

**Retraining schedule:** Monthly with rolling 90-day window  
**Trigger:** Immediate retraining if AUC drops > 0.03 in any week

## 5. ⚠️ Limitations & Risks

1. **Synthetic data:** PaySim is simulated — real-world performance may differ
2. **Concept drift:** Fraud patterns evolve; monthly retraining is essential
3. **Adversarial adaptation:** Fraudsters may learn to circumvent the model
4. **Regulatory requirements:** All blocked transactions require explainable reasoning (SHAP)
5. **Cold start:** New customer accounts lack historical balance data

## 6. ✅ Final Recommendations Summary

| Priority | Action | Timeline |
|---|---|---|
| 🔴 High | Deploy LightGBM in shadow mode alongside existing rules | Week 1-2 |
| 🔴 High | Set threshold to 0.30 for flag, 0.70 for block | Week 1-2 |
| 🟡 Medium | Build SHAP explanation service for analyst UI | Week 3-4 |
| 🟡 Medium | Set up weekly model performance dashboard | Week 3-4 |
| 🟢 Low | Train model on full transaction history (12+ months) | Month 2 |
| 🟢 Low | Implement graph-based features (transaction networks) | Month 3 |

---

## 🎯 Project Conclusion

This project demonstrated that:
- **LightGBM achieves ~0.99 ROC-AUC** on the PaySim fraud detection task
- **Engineered features** (balance discrepancies, account draining) are more predictive than raw features
- **SMOTE + class weighting** effectively addresses the 0.13% fraud rate imbalance
- **Threshold tuning** can increase fraud detection from 50% to 89%+ with manageable false alarm rates
- **Estimated net benefit:** Millions in prevented fraud loss per year at scale

*Project complete. See reports/dashboard.html for the interactive summary dashboard.*
